In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import numpy as np 

In [2]:
def collect_gmean_data_across_runs(base_path, abtype, window_length, param_val, algorithms_csv_names):
    """
    Collects G-Mean values across 20 runs for a specific pattern and window length.
    Returns a dictionary where keys are algorithm names and values are lists of G-Mean values.
    """
    data = {alg: [] for alg in algorithms_csv_names}
    
    # Construct the directory path for the specific parameter setting
    # Structure: base_path/abtype{X}/abtype{X}_w{W}_t{T}
    # Note: param_val might need formatting to match directory name exactly (e.g. 0.051 instead of 0.0510)
    # Based on user input, it seems to be just str(param_val) or formatted simply.
    # We will try to match the directory name.
    
    dir_name_parent = f"abtype{abtype}"
    dir_name_child = f"abtype{abtype}_w{window_length}_t{param_val}"
    target_dir = os.path.join(base_path, dir_name_parent, dir_name_child)
    
    if not os.path.exists(target_dir):
        print(f"Warning: Directory not found at {target_dir}.", file=sys.stderr)
        return data
        
    for alg_csv_name in algorithms_csv_names:
        for run_id in range(1, 21): # Runs 1 to 20
            file_name = f"{alg_csv_name}_run_{run_id}.csv"
            file_path = os.path.join(target_dir, file_name)
            
            if not os.path.exists(file_path):
                # print(f"Warning: File not found at {file_path}.", file=sys.stderr)
                continue
            
            try:
                df = pd.read_csv(file_path)
                # Filter for the last captured time (1000)
                # The user mentioned "Captured Time" goes up to 1000.
                df_filtered = df[df['Captured Time'] == 1000]
                
                if not df_filtered.empty:
                    # Assuming G-Mean is the column name
                    g_mean = df_filtered['G-Mean'].iloc[0]
                    data[alg_csv_name].append(g_mean)
                else:
                     print(f"Warning: No data for Captured Time 1000 in {file_path}", file=sys.stderr)
            
            except Exception as e:
                print(f"Error processing {file_path}: {e}", file=sys.stderr)
    
    return data

In [3]:
def create_boxplot_for_pattern_runs(base_path, pattern_config, algorithms_info, window_length=50):
    """
    Creates a box plot for a specific pattern showing G-Mean distributions across 20 runs.
    """
    # Collect data
    gmean_data = collect_gmean_data_across_runs(
        base_path,
        pattern_config['abtype'],
        window_length,
        pattern_config['param_val'],
        algorithms_info['csv_names']
    )
    
    # Prepare data for plotting
    plot_data = []
    plot_labels = []
    
    for latex_name, csv_name in zip(algorithms_info['latex_names'], algorithms_info['csv_names']):
        if gmean_data[csv_name]:  # Only include if we have data
            plot_data.append(gmean_data[csv_name])
            plot_labels.append(latex_name)
        else:
            print(f"Warning: No data found for {csv_name} in {pattern_config['pattern_name']}")
    
    if not plot_data:
        print(f"No data to plot for {pattern_config['pattern_name']}")
        return

    # Create the plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Create box plot
    bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                    showfliers=True,  # Show outliers
                    boxprops=dict(facecolor='cornflowerblue', color='black', linewidth=1.5),
                    whiskerprops=dict(color='black', linewidth=1.5),
                    capprops=dict(color='black', linewidth=1.5),
                    medianprops=dict(color='darkred', linewidth=2),
                    flierprops=dict(marker='D', markerfacecolor='gray', markersize=6, 
                                   markeredgecolor='black', alpha=0.7))
    
    # Set labels and title with size 20
    ax.set_xlabel('Algorithm', fontsize=20, fontweight='bold')
    ax.set_ylabel('G-Mean', fontsize=20, fontweight='bold')
    # ax.set_title(f"{pattern_config['pattern_name']} Pattern (Window Length {window_length})", fontsize=22, fontweight='bold', pad=20)
    
    # Set y-axis limits (Adjust as needed based on data range)
    ax.set_ylim(0.45, 0.68) 
    # Auto-scaling might be better initially to see the data range
    
    # Set tick parameters
    ax.tick_params(axis='x', labelsize=20, rotation=45)
    ax.tick_params(axis='y', labelsize=20)
    
    # Add grid for better readability
    ax.grid(True, axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    # Save the figure
    output_filename = f"boxplot_runs_{pattern_config['pattern_name'].lower().replace('-', '_')}.png"
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_filename}")
    
    plt.show()
    plt.close()

In [ ]:
def generate_all_run_boxplots():
    """
    Main function to generate box plots for all patterns.
    """
    # --- Configuration ---
    BASE_PATH = r'C:\Users\pault\Documents\3. AI and Machine Learning\2. Deep Learning\1c. App\Projects\CCPR_project\CCPR_2\results\individual_runs'
    
    # Algorithm mapping
    LATEX_TO_CSV_ALGO_MAP = {
        "PA": "PA",
        "PA-I": "PA1",
        "PA-II": "PA2",
        "CSPA$_1$": "PA1_Csplit",
        "CSPA$_2$": "PA2_Csplit",
        "CSPA-$\\ell$1": "PA_L1",
        "CSPA-$\\ell$2": "PA_L2",
        "CSPA$_1$-$\\ell^{I}$": "PA1_L1",
        "CSPA$_1$-$\\ell^{II}$": "PA1_L2",
        "CSPA$_2$-$\\ell^{I}$": "PA2_L1",
        "CSPA$_2$-$\\ell^{II}$": "PA2_L2",
    }
    
    algorithms_info = {
        'latex_names': list(LATEX_TO_CSV_ALGO_MAP.keys()),
        'csv_names': list(LATEX_TO_CSV_ALGO_MAP.values())
    }
    
    # Pattern configurations
    # Using Window Length 50 for all as requested
    PATTERN_CONFIGS = [
        {
            "pattern_name": "Up-trend",
            "abtype": 1,
            "param_val": 0.051,
        },
        {
            "pattern_name": "Down-trend",
            "abtype": 2,
            "param_val": 0.051,
        },
        {
            "pattern_name": "Up-shift",
            "abtype": 3,
            "param_val": 0.236,
        },
        {
            "pattern_name": "Down-shift",
            "abtype": 4,
            "param_val": 0.236,
        },
        {
            "pattern_name": "Systematic",
            "abtype": 5,
            "param_val": 0.282,
        },
        {
            "pattern_name": "Cyclic",
            "abtype": 6,
            "param_val": 0.42,
        },
    ]
    
    WINDOW_LENGTH = 50
    
    # Generate box plot for each pattern
    print("Generating box plots for all patterns (Variation across Runs)...\n")
    for pattern_config in PATTERN_CONFIGS:
        print(f"Creating box plot for {pattern_config['pattern_name']}...")
        create_boxplot_for_pattern_runs(BASE_PATH, pattern_config, algorithms_info, window_length=WINDOW_LENGTH)
    
    print("\nAll box plots generated successfully!")

In [7]:
if __name__ == '__main__':
    generate_all_run_boxplots()

Generating box plots for all patterns (Variation across Runs)...

Creating box plot for Up-trend...
No data to plot for Up-trend
Creating box plot for Down-trend...
No data to plot for Down-trend
Creating box plot for Up-shift...
No data to plot for Up-shift
Creating box plot for Down-shift...
No data to plot for Down-shift
Creating box plot for Systematic...
No data to plot for Systematic
Creating box plot for Cyclic...
No data to plot for Cyclic

All box plots generated successfully!
